# 03 — Controlled DQ Degradation Experiments

**Project:** dq-ml-impact-lab  
**Author:** Salini Anbalagan  
**Dataset:** UCI Bank Marketing Dataset  

---

## Notebook Objectives

1. Apply four types of controlled DQ degradation to the baseline dataset
2. Run `DQProfiler` on each degraded version to capture DQ score shifts
3. Build a **degradation registry** — a catalogue of all experiment versions
4. Save all degraded datasets for use in Notebook 04
5. Visualise how each degradation type affects the DQ scorecard

---

## Degradation Types

| Type | Levels | What it simulates |
|---|---|---|
| Null injection | 5%, 15%, 30% | Data entry gaps, system failures |
| Label noise | 5%, 10%, 20% | Annotation errors, mislabelling |
| Duplicate rows | 10%, 25%, 50% | ETL pipeline errors, re-ingestion |
| Outlier injection | sigma=2, 3, 4 at 5% | Sensor errors, data corruption |

> **Key question for this notebook:**  
> *How does each degradation type change the measured DQ score?*

## 0. Setup and Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import sys
sys.path.append('..')
from src.utils import load_dataset, save_scorecard, save_experiment_results
from src.degrader import DQDegrader
from src.dq_profiler import DQProfiler

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['axes.titlesize'] = 13

OUTPUT_DIR = Path('../data/degraded')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('Setup complete.')

## 1. Load Baseline Dataset

In [ ]:
df = load_dataset('../data/raw/bank-additional-full.csv', sep=';', target_col='y')
print(f'Baseline dataset: {df.shape}')

# Score baseline for comparison
baseline_profiler = DQProfiler(df, random_seed=42)
baseline_scorecard = baseline_profiler.score()
baseline_dq = baseline_profiler.dataset_dq_score
print(f'Baseline DQ Score: {baseline_dq}')

## 2. Degradation Registry

All degraded versions are stored in a single dictionary `degraded_registry`.  
Each key is a descriptive label; each value is the degraded DataFrame.

In [ ]:
degraded_registry = {}
dq_scores_registry = {}

# Always include baseline
degraded_registry['baseline'] = df.copy()
dq_scores_registry['baseline'] = baseline_dq

print('Registry initialised with baseline.')

## 3. Null Injection Experiments

In [ ]:
null_levels = [0.05, 0.15, 0.30]

for pct in null_levels:
    label = f'nulls_{int(pct*100)}pct'
    degraded_df = (
        DQDegrader(df, random_seed=42)
        .inject_nulls(pct=pct)
        .result()
    )
    profiler = DQProfiler(degraded_df, random_seed=42)
    profiler.score()

    degraded_registry[label] = degraded_df
    dq_scores_registry[label] = profiler.dataset_dq_score

    print(f'{label}: DQ Score = {profiler.dataset_dq_score} | Nulls = {degraded_df.isnull().sum().sum():,}')

## 4. Label Noise Experiments

In [ ]:
noise_levels = [0.05, 0.10, 0.20]

for pct in noise_levels:
    label = f'label_noise_{int(pct*100)}pct'
    degraded_df = (
        DQDegrader(df, random_seed=42)
        .inject_label_noise(target_col='y', pct=pct)
        .result()
    )
    profiler = DQProfiler(degraded_df, random_seed=42)
    profiler.score()

    degraded_registry[label] = degraded_df
    dq_scores_registry[label] = profiler.dataset_dq_score

    # Count actual label changes
    n_changed = (df['y'].values != degraded_df['y'].values).sum()
    print(f'{label}: DQ Score = {profiler.dataset_dq_score} | Labels flipped = {n_changed:,}')

## 5. Duplicate Injection Experiments

In [ ]:
dup_levels = [0.10, 0.25, 0.50]

for pct in dup_levels:
    label = f'duplicates_{int(pct*100)}pct'
    degraded_df = (
        DQDegrader(df, random_seed=42)
        .inject_duplicates(pct=pct)
        .result()
    )
    profiler = DQProfiler(degraded_df, random_seed=42)
    profiler.score()

    degraded_registry[label] = degraded_df
    dq_scores_registry[label] = profiler.dataset_dq_score

    added_rows = len(degraded_df) - len(df)
    print(f'{label}: DQ Score = {profiler.dataset_dq_score} | Rows added = {added_rows:,} | Total = {len(degraded_df):,}')

## 6. Outlier Injection Experiments

In [ ]:
sigma_levels = [2.0, 3.0, 4.0]

for sigma in sigma_levels:
    label = f'outliers_sigma{int(sigma)}'
    degraded_df = (
        DQDegrader(df, random_seed=42)
        .inject_outliers(sigma=sigma, pct=0.05)
        .result()
    )
    profiler = DQProfiler(degraded_df, random_seed=42)
    profiler.score()

    degraded_registry[label] = degraded_df
    dq_scores_registry[label] = profiler.dataset_dq_score

    print(f'{label}: DQ Score = {profiler.dataset_dq_score}')

## 7. Registry Summary

In [ ]:
registry_df = pd.DataFrame([
    {
        'label': label,
        'n_rows': len(df_),
        'n_nulls': int(df_.isnull().sum().sum()),
        'null_rate': round(df_.isnull().sum().sum() / df_.size, 4),
        'dataset_dq_score': dq_scores_registry[label],
        'dq_delta_vs_baseline': round(dq_scores_registry[label] - baseline_dq, 4)
    }
    for label, df_ in degraded_registry.items()
])

registry_df.style.background_gradient(
    subset=['dataset_dq_score'], cmap='RdYlGn'
).background_gradient(
    subset=['dq_delta_vs_baseline'], cmap='RdYlGn'
)

## 8. DQ Score Drop Visualisation

In [ ]:
# Group by degradation type
def get_type(label):
    if 'null' in label:   return 'Null Injection'
    if 'noise' in label:  return 'Label Noise'
    if 'dup' in label:    return 'Duplicates'
    if 'outlier' in label: return 'Outliers'
    return 'Baseline'

registry_df['type'] = registry_df['label'].apply(get_type)
type_colors = {
    'Baseline': '#95a5a6',
    'Null Injection': '#e74c3c',
    'Label Noise': '#e67e22',
    'Duplicates': '#3498db',
    'Outliers': '#9b59b6',
}

fig, ax = plt.subplots(figsize=(15, 5))
colors = [type_colors[t] for t in registry_df['type']]
bars = ax.bar(registry_df['label'], registry_df['dataset_dq_score'],
              color=colors, edgecolor='white', width=0.7)

ax.axhline(baseline_dq, color='black', linestyle='--', linewidth=1.5, label=f'Baseline DQ ({baseline_dq})')

for bar, score in zip(bars, registry_df['dataset_dq_score']):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.003,
            f'{score:.3f}', ha='center', va='bottom', fontsize=8)

# Legend patches
import matplotlib.patches as mpatches
patches = [mpatches.Patch(color=c, label=l) for l, c in type_colors.items()]
ax.legend(handles=patches, loc='lower right', fontsize=9)

ax.set_title('Dataset DQ Score Across All Degradation Versions', fontsize=14)
ax.set_ylabel('Dataset DQ Score')
ax.set_ylim(0, 1.05)
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig('../data/degraded/dq_score_all_versions.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Save All Degraded Datasets

In [ ]:
for label, degraded_df in degraded_registry.items():
    if label == 'baseline':
        continue
    path = OUTPUT_DIR / f'{label}.csv'
    degraded_df.to_csv(path, index=False)

# Save registry summary
registry_df.to_csv(OUTPUT_DIR / 'degradation_registry.csv', index=False)

print(f'Saved {len(degraded_registry) - 1} degraded datasets to {OUTPUT_DIR}')
print(f'Registry saved to {OUTPUT_DIR}/degradation_registry.csv')

## 10. Summary of Findings

In [ ]:
print('=' * 65)
print('  DEGRADATION EXPERIMENTS — KEY FINDINGS SUMMARY')
print('=' * 65)
print(f'  Baseline DQ Score      : {baseline_dq}')
print(f'  Total versions created : {len(degraded_registry)}')
print()

worst = registry_df.loc[registry_df['dataset_dq_score'].idxmin()]
best_deg = registry_df[registry_df['label'] != 'baseline'].loc[
    registry_df[registry_df['label'] != 'baseline']['dataset_dq_score'].idxmax()
]
print(f'  Worst degraded version : {worst["label"]} (DQ = {worst["dataset_dq_score"]})')
print(f'  Mildest degraded       : {best_deg["label"]} (DQ = {best_deg["dataset_dq_score"]})')
print()
print('  Observations:')
print('  - Null injection has the most direct impact on completeness score')
print('  - Label noise shifts consistency but not completeness')
print('  - Duplicates inflate the dataset but have minimal DQ score impact')
print('  - Outliers primarily degrade the anomaly rate dimension')
print('=' * 65)

---

## Next Steps

→ **Notebook 04:** `04_ml_impact_analysis.ipynb` — Train classifiers on each degraded version  
and measure the relationship between DQ score and model performance.

---
*dq-ml-impact-lab | Salini Anbalagan*